# Character-level Diffusion-LM

A compact, corrected continuous embedding diffusion model. It follows the `x_0` parameterization and nearest-embedding clamping described in Diffusion-LM (Li et al., 2022).

Key choices: fixed-length contiguous character blocks (no padding distribution), timesteps in `1..T`, direct clean-embedding prediction, and a DDIM reverse sampler.

In [20]:
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(7)
random.seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

mps


In [21]:
# Use fixed blocks so generation is trained on exactly the same shape it sees at inference.
DATA_DIR = Path('data/shakespeare-dataset-main/text')
text = '\n'.join(path.read_text(encoding='utf-8') for path in sorted(DATA_DIR.glob('*.txt')))

# Keep every observed character: this avoids silently converting rare characters to <UNK>.
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
tokens = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

class CharacterBlocks(Dataset):
    def __init__(self, values, block_size=64, stride=None):
        self.values = values
        self.block_size = block_size
        self.stride = stride or block_size
        self.starts = range(0, len(values) - block_size + 1, self.stride)
    def __len__(self):
        return len(self.starts)
    def __getitem__(self, index):
        start = self.starts[index]
        return self.values[start:start + self.block_size]

BLOCK_SIZE = 64
dataset = CharacterBlocks(tokens, BLOCK_SIZE)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True, num_workers=0)
print(f'{len(text):,} characters | vocab: {vocab_size} | blocks: {len(dataset):,}')

5,330,377 characters | vocab: 84 | blocks: 83,287


In [22]:
# alpha_bar[t] is the cumulative signal coefficient for x_t.
# Index 0 is exactly clean data; train only on t = 1..T.
def cosine_schedule(T, s=0.008):
    steps = torch.arange(T + 1, dtype=torch.float32)
    alpha_bar = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2).square()
    alpha_bar = alpha_bar / alpha_bar[0]
    return alpha_bar.clamp(min=1e-6, max=1.0)

T = 1000
alpha_bar = cosine_schedule(T).to(device)
assert torch.isclose(alpha_bar[0], torch.tensor(1.0, device=device))
print(alpha_bar[0].item(), alpha_bar[-1].item())

1.0 9.999999974752427e-07


In [23]:
def timestep_embedding(t, dim):
    half = dim // 2
    frequencies = torch.exp(-math.log(10_000) * torch.arange(half, device=t.device) / max(half - 1, 1))
    embedding = t.float().unsqueeze(1) * frequencies.unsqueeze(0)
    embedding = torch.cat((embedding.sin(), embedding.cos()), dim=1)
    return F.pad(embedding, (0, dim % 2))

class DenoiserBlock(nn.Module):
    def __init__(self, dim, heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, dim))
    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.ff(self.norm2(x))

class EmbeddingDiffusionLM(nn.Module):
    def __init__(self, vocab_size, dim=256, heads=4, ff_dim=1024, layers=6, max_length=64):
        super().__init__()
        self.dim = dim
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(max_length, dim)
        self.time_mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))
        self.blocks = nn.ModuleList([DenoiserBlock(dim, heads, ff_dim) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(dim)
        self.output = nn.Linear(dim, dim)
        # This is the paper's separate p(w | x0) rounding model.
        # It is intentionally not applied to noisy denoiser predictions.
        self.rounding_head = nn.Linear(dim, vocab_size)
    def forward(self, x_t, t):
        batch, length, _ = x_t.shape
        positions = torch.arange(length, device=x_t.device)
        time = self.time_mlp(timestep_embedding(t, self.dim)).unsqueeze(1)
        x = x_t + self.position_embedding(positions).unsqueeze(0) + time
        for block in self.blocks:
            x = block(x)
        return self.output(self.final_norm(x))  # predicts x_0 directly

model = EmbeddingDiffusionLM(vocab_size, max_length=BLOCK_SIZE).to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

5,389,908 parameters


In [31]:
def token_logits(vectors, embedding_weight):
    # -||v - e||^2, omitting -||v||^2 because it is constant over vocabulary entries.
    return 2 * vectors @ embedding_weight.T - embedding_weight.square().sum(dim=1)

# Trainable part of KL[N(E[w], sigma_0^2 I) || N(0, I)] when sigma_0 is fixed.
# This discourages a degenerate large-norm codebook that leaks token identity at high noise.
EMBEDDING_KL_WEIGHT = 2e-3  # try 0.0, 1e-5, 5e-5, and 1e-4 as an ablation

def embedding_kl_mean_term(embedding_weight):
    return 0.5 * embedding_weight.square().sum(dim=1).mean()

def diffusion_loss(token_batch):
    token_batch = token_batch.to(device)
    clean_embeddings = model.token_embedding(token_batch)
    # Diffusion-LM samples x0 around a token embedding. This prevents a brittle exact lookup.
    x0 = clean_embeddings + 0.01 * torch.randn_like(clean_embeddings)
    t = torch.randint(1, T + 1, (token_batch.size(0),), device=device)
    abar = alpha_bar[t].view(-1, 1, 1)
    eps = torch.randn_like(x0)
    x_t = abar.sqrt() * x0 + (1 - abar).sqrt() * eps

    x0_hat = model(x_t, t)
    mse = F.mse_loss(x0_hat, x0)
    # Rounding is learned on clean x0 only. Applying CE to x0_hat at every t
    # forces unjustified high-confidence token predictions at high-noise timesteps.
    rounding_logits = model.rounding_head(x0)
    rounding = F.cross_entropy(rounding_logits.flatten(0, 1), token_batch.flatten())
    embedding_kl = embedding_kl_mean_term(model.token_embedding.weight)
    loss = mse + rounding + EMBEDDING_KL_WEIGHT * embedding_kl
    return loss, mse.detach(), rounding.detach(), embedding_kl.detach()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)

def train_epoch():
    model.train()
    totals = torch.zeros(4)
    for batch in loader:
        loss, mse, ce, embedding_kl = diffusion_loss(batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        totals += torch.tensor([loss.item(), mse.item(), ce.item(), embedding_kl.item()])
    return (totals / len(loader)).tolist()

# Run a few epochs first; do not evaluate free samples until reconstruction improves.
for epoch in range(1, 10):
    loss, mse, ce, embedding_kl = train_epoch()
    print(f'epoch {epoch}: loss={loss:.4f}, x0_mse={mse:.4f}, token_ce={ce:.4f}, embedding_kl={embedding_kl:.2f}')

epoch 1: loss=0.0561, x0_mse=0.0312, token_ce=0.0006, embedding_kl=12.14
epoch 2: loss=0.0334, x0_mse=0.0200, token_ce=0.0005, embedding_kl=6.43
epoch 3: loss=0.0199, x0_mse=0.0124, token_ce=0.0005, embedding_kl=3.53
epoch 4: loss=0.0118, x0_mse=0.0074, token_ce=0.0004, embedding_kl=1.99
epoch 5: loss=0.0070, x0_mse=0.0043, token_ce=0.0004, embedding_kl=1.15
epoch 6: loss=0.0043, x0_mse=0.0026, token_ce=0.0003, embedding_kl=0.69
epoch 7: loss=0.0029, x0_mse=0.0018, token_ce=0.0003, embedding_kl=0.43
epoch 8: loss=0.0021, x0_mse=0.0013, token_ce=0.0002, embedding_kl=0.29
epoch 9: loss=0.0016, x0_mse=0.0010, token_ce=0.0002, embedding_kl=0.21


In [25]:
@torch.no_grad()
def denoising_check(token_batch, t_value=500):
    model.eval()
    ids = token_batch[:1].to(device)
    x0 = model.token_embedding(ids)
    eps = torch.randn_like(x0)
    abar = alpha_bar[t_value]
    x_t = abar.sqrt() * x0 + (1 - abar).sqrt() * eps
    x0_hat = model(x_t, torch.tensor([t_value], device=device))
    recovered = token_logits(x0_hat, model.token_embedding.weight).argmax(-1)
    accuracy = (recovered == ids).float().mean().item()
    print(f't={t_value}, token recovery={accuracy:.1%}')
    print('target:   ', repr(decode(ids[0].cpu())))
    print('recovered:', repr(decode(recovered[0].cpu())))

check_batch = next(iter(loader))
for t_value in (50, 500, 900, 999):
    denoising_check(check_batch, t_value=t_value)

# If this grows very large, x_T is no longer close to N(0, I), which makes free sampling fail.
embedding_norms = model.token_embedding.weight.norm(dim=1)
print(f'embedding norm: mean={embedding_norms.mean():.3f}, max={embedding_norms.max():.3f}')

t=50, token recovery=100.0%
target:    'who\nreads it.)] But let thine inherit first, for I protest\nmine '
recovered: 'who\nreads it.)] But let thine inherit first, for I protest\nmine '
t=500, token recovery=98.4%
target:    'who\nreads it.)] But let thine inherit first, for I protest\nmine '
recovered: 'who\nreads it.:] But let thine inherit first, for I protest\nmine '
t=900, token recovery=20.3%
target:    'who\nreads it.)] But let thine inherit first, for I protest\nmine '
recovered: ' h2\t82\t2s  t2a\t\t22t222  22i2\t 22s\th t22\t22\t2 822o22\t222\t 2\n28\te '
t=999, token recovery=0.0%
target:    'who\nreads it.)] But let thine inherit first, for I protest\nmine '
recovered: '2222222222222222222222222222222222222222222222222222222222222222'
embedding norm: mean=5.718, max=8.932


In [49]:
@torch.no_grad()
def clamp_to_embeddings(x0_hat):
    ids = token_logits(x0_hat, model.token_embedding.weight).argmax(-1)
    return model.token_embedding(ids), ids

@torch.no_grad()
def reverse_diffuse(x_t, start_t, steps=100, eta=0.0, clamp_from_t=800):
    """Reverse a supplied x_t to x_0 with DDIM. eta=0 is deterministic."""
    model.eval()
    times = torch.linspace(start_t, 0, steps + 1, device=device).round().long().unique_consecutive()
    for t, s in zip(times[:-1], times[1:]):
        t_batch = t.expand(1)
        abar_t = alpha_bar[t]
        abar_s = alpha_bar[s]
        x0_hat = model(x_t, t_batch)
        # The paper's clamping trick: nearest embedding before the next diffusion state.
        x0_used, _ = clamp_to_embeddings(x0_hat) if t <= clamp_from_t else (x0_hat, None)
        # Once x0 is clamped, recompute epsilon from that same x0.
        # Otherwise the DDIM direction and the leading x0 term describe different trajectories.
        eps_hat = (x_t - abar_t.sqrt() * x0_used) / (1 - abar_t).clamp_min(1e-6).sqrt()
        sigma = eta * (((1 - abar_s) / (1 - abar_t)) * (1 - abar_t / abar_s)).clamp_min(0).sqrt()
        direction = (1 - abar_s - sigma.square()).clamp_min(0).sqrt() * eps_hat
        x_t = abar_s.sqrt() * x0_used + direction
        if s > 0 and eta > 0:
            x_t = x_t + sigma * torch.randn_like(x_t)
    _, ids = clamp_to_embeddings(x_t)
    return ids

@torch.no_grad()
def sample(length=64, steps=100, eta=0.0, clamp_from_t=800):
    ids = reverse_diffuse(torch.randn(1, length, model.dim, device=device), T, steps, eta, clamp_from_t)
    return decode(ids[0].cpu())

@torch.no_grad()
def reverse_reconstruction_check(token_batch, start_t=900, steps=100, clamp_from_t=800):
    # This test starts from a legitimately noised target, not from N(0, I).
    # If it fails while direct denoising succeeds, the reverse update is at fault.
    ids = token_batch[:1].to(device)
    x0 = model.token_embedding(ids)
    abar = alpha_bar[start_t]
    x_t = abar.sqrt() * x0 + (1 - abar).sqrt() * torch.randn_like(x0)
    recovered = reverse_diffuse(x_t, start_t, steps, eta=0.0, clamp_from_t=clamp_from_t)
    print(f'reverse recovery from t={start_t}: {(recovered == ids).float().mean().item():.1%}')
    print('target:   ', repr(decode(ids[0].cpu())))
    print('recovered:', repr(decode(recovered[0].cpu())))

reverse_reconstruction_check(next(iter(loader)), start_t=50)

#print(sample(length=64, steps=1000, eta=0.0))

reverse recovery from t=50: 98.4%
target:    "nelaus' queen,\nWith wanton Paris sleeps; and that's the quarrel."
recovered: "nelaus' jueen,\nWith wanton Paris sleeps; and that's the quarrel."
